# Thống kê mô tả dữ liệu CafeLand
Notebook tổng quan nhanh về kiểu dữ liệu, khoảng giá trị, tỷ lệ thiếu và phân bố một số biến phân loại để chuẩn bị cho bước xử lý dữ liệu.

## 1. Nạp dữ liệu đã làm sạch

In [1]:
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
data_relative_path = Path("data") / "processed" / "cafeland_cleaned.csv"
candidate_roots = [current_dir, current_dir / "Project", current_dir.parent]
project_root = next((root for root in candidate_roots if (root / data_relative_path).exists()), None)
data_path = project_root / data_relative_path if project_root else None
if data_path is None:
    raise FileNotFoundError(f"Không tìm thấy dữ liệu đã làm sạch từ thư mục hiện tại: {current_dir}")

df = pd.read_csv(data_path, encoding="utf-8-sig")
print(f"Đã nạp {len(df):,} dòng và {len(df.columns)} cột từ {data_path.name}.")
display(df.head())

Đã nạp 6,500 dòng và 9 cột từ cafeland_cleaned.csv.


,tieu_de,gia,dien_tich,vi_tri,ngay_dang,link_nguon,mo_ta,nguoi_ban,so_phong_ngu
0,KẸT TIỀN CẦN BÁN 2PN+ 72M TẦNG 20+ MAISON GRAN...,2.990000e+09,72.0,"Phú Mỹ, Thành phố Hồ Chí Minh",2026-09-16 09:17:35,https://nhadat.cafeland.vn/ket-tien-can-ban-2p...,**2PN+ 72M² TẦNG CAO – MAISON GRAND PHÚ MỸ | G...,Phạm Thị Quyên,2.0
1,"BÁN NHÀ MỚI KENG MẶT TIỀN TÂN XUÂN, HÓC MÔN ,S...",5.350000e+09,100.0,"An Lạc, Thành phố Hồ Chí Minh",2026-09-16 09:06:35,https://nhadat.cafeland.vn/ban-nha-moi-keng-ma...,"BÁN NHÀ MỚI KENG MẶT TIỀN ĐƯỜNG KINH DOANH,ẤP ...",Trần Ngọc Tuấn,3.0
2,"ĐẤT 1/NGUYỄN THỊ SÁNG , ĐÔNG THẠNH ,HÓC MÔN 5M...",3.750000e+09,105.0,"Chợ Lớn, Thành phố Hồ Chí Minh",2026-09-16 09:06:29,https://nhadat.cafeland.vn/dat-1nguyen-thi-san...,"HÀNG F0 MỚI RA LÒ , ĐẤT 1/NGUYỄN THỊ SÁNG , ĐÔ...",Trần Ngọc Tuấn,NaN
3,CHỦ CẦN BÁN GẤP GIẢM MẠNH 6 TỶ BÁN NHÀ MẶT TIỀ...,3.900000e+10,454.0,"Hóc Môn, Thành phố Hồ Chí Minh",2026-09-16 09:06:24,https://nhadat.cafeland.vn/chu-can-ban-gap-gia...,CHỦ CẦN BÁN GẤP GIẢM MẠNH 6 TỶ BÁN NHÀ MẶT TIỀ...,Trần Ngọc Tuấn,NaN
4,BÁN ĐẤT MẶT TIỀN ĐƯỜNG NHỰA DIỆN TÍCH KHỦNG 1/...,1.800000e+10,1270.3,"Hóc Môn, Thành phố Hồ Chí Minh",2026-09-16 09:06:18,https://nhadat.cafeland.vn/ban-dat-mat-tien-du...,BÁN ĐẤT MẶT TIỀN ĐƯỜNG NHỰA DIỆN TÍCH KHỦNG 1/...,Tuấnland88,NaN


## 2. Kích thước và kiểu dữ liệu

In [2]:
print(f"Số dòng: {df.shape[0]:,} | Số cột: {df.shape[1]}")
display(df.dtypes.rename("Kiểu dữ liệu").to_frame())

Số dòng: 6,500 | Số cột: 9


,Kiểu dữ liệu
tieu_de,object
gia,float64
dien_tich,float64
vi_tri,object
ngay_dang,object
link_nguon,object
mo_ta,object
nguoi_ban,object
so_phong_ngu,float64


## 3. Thống kê mô tả các biến số
Bảng `describe()` cung cấp số lượng quan sát hợp lệ, trung bình, độ lệch chuẩn, các phân vị cùng giá trị nhỏ nhất (Min) và lớn nhất (Max).

In [3]:
numeric_columns = df.select_dtypes(include="number").columns
if len(numeric_columns) == 0:
    print("Không có cột kiểu số để thống kê.")
else:
    display(df[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
gia,6008.0,3.198635e+10,2.071517e+11,1.48,3.741500e+09,7.663000e+09,1.850000e+10,7.140000e+12
dien_tich,4718.0,4.829983e+03,2.638654e+05,2.00,6.000000e+01,9.000000e+01,1.760000e+02,1.809000e+07
so_phong_ngu,2753.0,4.192154e+00,7.780635e+00,1.00,2.000000e+00,2.000000e+00,4.000000e+00,1.060000e+02


## 4. Tỷ lệ dữ liệu thiếu
Tỷ lệ được tính trên tổng số dòng của từng cột; bảng được sắp xếp từ cột thiếu nhiều nhất.

In [4]:
missing_summary = pd.DataFrame({
    "Số giá trị thiếu": df.isna().sum(),
    "Tỷ lệ thiếu (%)": (df.isna().mean() * 100).round(2),
}).sort_values("Tỷ lệ thiếu (%)", ascending=False)

display(missing_summary)

,Số giá trị thiếu,Tỷ lệ thiếu (%)
so_phong_ngu,3747,57.65
dien_tich,1782,27.42
gia,492,7.57
tieu_de,0,0.00
vi_tri,0,0.00
ngay_dang,0,0.00
link_nguon,0,0.00
mo_ta,0,0.00
nguoi_ban,0,0.00


## 5. Min, Max và tỷ lệ thiếu để bàn giao xử lý
Bảng gọn dưới đây tổng hợp các tham số cần xem xét cho từng biến số. Các giá trị ngoại lệ cần được kiểm tra theo ngữ cảnh trước khi quyết định xử lý.

In [5]:
numeric_handoff = pd.DataFrame({
    "Min": df[numeric_columns].min(),
    "Max": df[numeric_columns].max(),
    "Số giá trị thiếu": df[numeric_columns].isna().sum(),
    "Tỷ lệ thiếu (%)": (df[numeric_columns].isna().mean() * 100).round(2),
})
display(numeric_handoff)

,Min,Max,Số giá trị thiếu,Tỷ lệ thiếu (%)
gia,1.48,7.140000e+12,492,7.57
dien_tich,2.00,1.809000e+07,1782,27.42
so_phong_ngu,1.00,1.060000e+02,3747,57.65


## 6. Tần suất các biến phân loại
Hiển thị 10 giá trị phổ biến nhất cho `vi_tri` và `nguoi_ban`; giá trị thiếu vẫn được tính trong kết quả.

In [6]:
for column in ["vi_tri", "nguoi_ban"]:
    if column in df.columns:
        print(f"--- Top 10 giá trị của {column} ---")
        display(df[column].value_counts(dropna=False).head(10).rename_axis(column).to_frame("Số tin đăng"))
    else:
        print(f"Không tìm thấy cột {column!r} trong dữ liệu.")

--- Top 10 giá trị của vi_tri ---


,Số tin đăng
vi_tri,
"An Khánh, Thành phố Hồ Chí Minh",383
"Bình Dương, Thành phố Hồ Chí Minh",337
"Lái Thiêu, Thành phố Hồ Chí Minh",301
"Tân Hưng, Thành phố Hồ Chí Minh",291
"Dĩ An, Thành phố Hồ Chí Minh",185
"Cần Giờ, Thành phố Hồ Chí Minh",151
"Bàn Cờ, Thành phố Hồ Chí Minh",145
"Hiệp Bình, Thành phố Hồ Chí Minh",135
"Bình Trị Đông, Thành phố Hồ Chí Minh",131


--- Top 10 giá trị của nguoi_ban ---


,Số tin đăng
nguoi_ban,
Mai Phương,408
Duy Thanh,210
Trần Duy Trường,177
Nguyễn Quốc Mạnh Tuấn,169
lương quốc thanh,161
Phạm Tấn Sơn,112
Dũng La Pura,108
Đỗ Thuý Nga,96
NGUYỄN ĐỨC ANH,92
